# Lasso Regression: From First Principles

## 1. Why Do We Need Lasso?

Suppose we predict house prices using floor area, age, bedroom count, neighborhood indicators, and dozens of other measurements. Some features may add useful information; others may repeat information already present or mostly capture noise. Ordinary Linear Regression fits all supplied coefficients without a penalty. With limited data, this can produce a model that fits accidental patterns too closely.

Ridge addresses instability by discouraging large coefficients, but generally keeps each feature in the prediction equation. Sometimes we also want a model that uses fewer features. **Lasso** adds a different penalty that can set some coefficients exactly to zero. Its name comes from *least absolute shrinkage and selection operator*.

The original notes compare this to packing for a trip: Ridge encourages a lighter bag, while Lasso can leave items behind. The analogy is useful, but the algorithm does not know which features truly matter. It balances training fit against coefficient size using the available data.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Core intuition</strong><br>Lasso allows a feature to contribute only when the improvement in fit justifies the penalty on its coefficient. This can produce a sparse prediction equation, with many coefficients equal to zero.</div>

We will build the objective, explain how zeros arise, fit a small example, and then connect the method to preprocessing, validation, and the companion notebook.

## 2. What Lasso Learns

Lasso is a **supervised regression** method. Training rows contain features and known numerical targets. A fitted model receives the features of a new row and returns a numerical prediction. It does not directly produce class probabilities.

For $n$ training rows and $p$ features, the prediction for row $i$ is

$$
\hat y_i=b+\sum_{j=1}^{p}w_jx_{ij}.
$$

| Symbol or term | Meaning |
|---|---|
| $x_{ij}$ | Value of feature $j$ for row $i$ |
| $y_i$ and $\hat y_i$ | Observed target and predicted target |
| $w_j$ | Learned coefficient for feature $j$ |
| $b$ | Learned intercept, the prediction when all represented features are zero |
| Residual $e_i=y_i-\hat y_i$ | Observed value minus prediction |
| Sparse model | A model with many coefficients exactly zero |
| Selected features | Features whose fitted coefficients are nonzero |

The coefficients and intercept are fitted **parameters**. The penalty strength is a **hyperparameter** chosen through validation. A zero coefficient makes that feature's contribution zero in this fitted equation. It does not prove that the feature is irrelevant in every dataset or model.

The prediction rule remains linear in the supplied features. Nonlinear transformations, such as area squared, can be supplied as additional features, but Lasso does not invent them automatically.

## 3. The Objective: Fit Plus an L1 Penalty

Ordinary least squares minimizes squared residuals. Lasso adds the sum of absolute coefficient values:

$$
J(b,\mathbf w)=\frac{1}{2n}\sum_{i=1}^{n}\left(y_i-b-\sum_{j=1}^{p}w_jx_{ij}\right)^2
+\alpha\sum_{j=1}^{p}|w_j|,\qquad\alpha\geq0.
$$

The vector $\mathbf w$ contains all feature coefficients. The first term is half the mean squared error. The factor $1/2$ simplifies derivatives; the second term is the penalty. We use this convention to match the companion's scikit-learn `Lasso` estimator.

The **L1 norm** is $\|\mathbf w\|_1=\sum_j|w_j|$. Absolute values keep positive and negative coefficients from canceling. For coefficients $(5,-3,0,2)$, the L1 norm is $5+3+0+2=10$. Multiplying it by $\alpha$ determines its cost in the fitting objective.

The intercept is not penalized. It represents the overall baseline, while the penalty controls feature effects. During prediction, we evaluate the linear equation; we do not add or subtract a regularization penalty from the predicted price.

L1 is not a direct count of nonzero coefficients. A single coefficient of 100 has a larger L1 norm than ten coefficients of 1. Sparsity is an effect of this optimization problem, not an explicit instruction to keep a fixed number of features.

## 4. Reconciling Alpha and Lambda

The original notes write $\mathrm{MSE}+\lambda\sum_j|w_j|$. Our convention is $\tfrac12\mathrm{MSE}+\alpha\sum_j|w_j|$. Multiplying the latter by 2 gives the same minimizer and yields

$$
\mathrm{MSE}+2\alpha\sum_j|w_j|.
$$

For identical unweighted data and preprocessing, the conventions therefore agree when $\lambda=2\alpha$. If a text instead uses summed squared error plus a penalty, its penalty multiplier is $2n\alpha$.

The [scikit-learn Lasso documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) specifies the half-MSE objective and coordinate-descent solver. A numeric alpha in Lasso is also not directly comparable to the same numeric alpha in Ridge: both the loss normalization and penalty differ.

At $\alpha=0$, the objective is ordinary least squares. For an unregularized scikit-learn baseline, use `LinearRegression`, as the companion does, rather than using the Lasso solver with zero alpha.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Common pitfall</strong><br>Compare full objective functions before comparing penalty values. Alpha depends on the loss convention, feature representation, and target scale.</div>

## 5. First Compare Two Candidate Models

Retain the original notes' simple comparison, now with an explicit penalty calculation. Assume both candidates use the same feature representation and target units:

| Candidate | MSE | Coefficients | L1 norm |
|---|---:|---|---:|
| A | 12 | 5, 0, 0, 2 | 7 |
| B | 10 | 5, 3, 4, 2 | 14 |

With $\alpha=0.5$ under our half-MSE convention:

$$
J_A=12/2+0.5(7)=9.5,\qquad J_B=10/2+0.5(14)=12.
$$

B has smaller training error, but A has the smaller penalized objective. B's error improvement is not enough to offset its larger coefficient total. With $\alpha=0.1$, the values become $6.7$ and $6.4$, so B is preferred among these candidates.

This shows how the preference changes with alpha. It does not find the optimum over all possible coefficients or show which candidate predicts unseen houses better. Next we need to understand the actual fitting rule.

## 6. Why Lasso Can Produce Exact Zeros

Consider one centered feature $z_i$ and centered target $v_i$. Centering subtracts each training mean, so the intercept can be handled separately. Define

$$
a=\frac{1}{n}\sum_i z_i^2,\qquad c=\frac{1}{n}\sum_i z_iv_i.
$$

The quantity $a$ measures feature variation; $c$ measures how the feature and target move together. Expanding the objective and ignoring terms that do not depend on its coefficient $w$ gives

$$
J(w)=\frac{a}{2}w^2-cw+\alpha|w|+\text{constant}.
$$

For positive $w$, the derivative is $aw-c+\alpha$. Setting it to zero gives $w=(c-\alpha)/a$, which is positive only when $c>\alpha$. For negative $w$, the derivative is $aw-c-\alpha$, giving $w=(c+\alpha)/a$ when $c<-\alpha$.

At zero, absolute value has a corner instead of a single derivative. Zero is optimal when the slope from the left is nonpositive and the slope from the right is nonnegative. These conditions combine to $|c|\leq\alpha$.

For $a>0$, the full solution is

$$
w=\begin{cases}
(c-\alpha)/a,&c>\alpha,\\
0,&|c|\leq\alpha,\\
(c+\alpha)/a,&c<-\alpha.
\end{cases}
$$

This is **soft thresholding**: subtract a threshold from the magnitude of a sufficiently strong contribution, and set weaker contributions exactly to zero. If the feature is standardized so $a=1$, the denominator disappears. A constant centered feature has $a=0$ and supplies no predictive variation; for positive alpha its coefficient should be zero.

Ridge's squared penalty has a derivative that approaches zero as the coefficient approaches zero. Lasso's absolute-value penalty keeps a nonzero slope on either side of zero, creating this threshold region.

## 7. From One Coefficient to Many: Coordinate Descent

With several correlated features, each feature's usefulness depends on what the others already explain. We cannot independently threshold every feature's original relationship with the target and expect the joint solution.

**Coordinate descent** updates one coefficient at a time while holding the others fixed. Suppose features and targets have been centered. For feature $j$, calculate the partial residual

$$
r_i^{(j)}=v_i-\sum_{k\ne j}w_kz_{ik}.
$$

Here $z_{ik}$ is centered feature $k$ for row $i$, and $v_i$ is its centered target. This residual removes contributions from every feature except $j$, leaving the part that feature $j$ can try to explain. Then compute

$$
a_j=\frac{1}{n}\sum_i z_{ij}^2,\qquad c_j=\frac{1}{n}\sum_i z_{ij}r_i^{(j)}.
$$

Apply the previous soft-threshold rule using $a_j$ and $c_j$. The fitting sequence is:

1. Initialize the coefficients, for example at zero.
2. Calculate a feature's partial residual using current coefficients.
3. Update that coefficient with soft thresholding.
4. Repeat for the other features, then make additional passes until the convergence criterion is satisfied.

For centered features, recover the intercept as $b=\bar y-\sum_jw_j\bar x_j$, where bars denote training means in the feature representation being fitted. Standardized features also have zero training means.

The objective is convex, meaning it has no inferior local minima. Coefficient solutions can nevertheless be nonunique when features are redundant. Numerical convergence also matters: stopping an iterative solver too early can leave an inaccurate solution.

## 8. Worked Example: One Feature Survives

Use four hypothetical houses with two already standardized numerical features. Let $z_1$ describe floor area and $z_2$ describe an age-related measurement. Prices are in tens of thousands of dollars. The small, deliberately structured data make every calculation visible.

| House | $z_1$ | $z_2$ | Price $y$ | Centered price $v=y-30$ |
|---|---:|---:|---:|---:|
| A | -1 | -1 | 27 | -3 |
| B | -1 | 1 | 29 | -1 |
| C | 1 | -1 | 31 | 1 |
| D | 1 | 1 | 33 | 3 |

Both features have mean zero and mean square one. Their cross-product sum is $1-1-1+1=0$, so the columns are **orthogonal**: they do not overlap in this training sample. Consequently, their coefficient calculations separate.

For the first feature:

$$
a_1=1,\qquad c_1=\frac{3+1+1+3}{4}=2.
$$

For the second feature:

$$
a_2=1,\qquad c_2=\frac{3-1-1+3}{4}=1.
$$

Choose $\alpha=1$. Soft thresholding gives $w_1=(2-1)/1=1$ and $w_2=0$, because $|c_2|\leq1$. The intercept is $b=30$. Thus

$$
\hat y=30+z_1.
$$

Ordinary least squares gives $\hat y=30+2z_1+z_2$ and fits these four rows exactly. Lasso deliberately shrinks one coefficient and removes the other.

## 9. Check the Fit, Objective, and New Prediction

For the Lasso model from the previous section:

| House | Actual price | Prediction | Residual | Squared residual |
|---|---:|---:|---:|---:|
| A | 27 | 29 | -2 | 4 |
| B | 29 | 29 | 0 | 0 |
| C | 31 | 31 | 0 | 0 |
| D | 33 | 31 | 2 | 4 |

The SSE is 8, the MSE is $8/4=2$, and the L1 norm is 1. Therefore

$$
J_{\mathrm{lasso}}=\frac{8}{2(4)}+1(1)=2.
$$

Evaluating the OLS coefficients under the same penalized objective gives $0+1(2+1)=3$. An intercept-only model predicts 30 for every row, giving SSE $9+1+1+9=20$ and objective $20/8=2.5$. Lasso has the smallest objective of these three, consistent with the derived optimum.

A new house with standardized features $(0.5,-0.5)$ receives prediction $30+0.5=30.5$, or $305{,}000$. The second feature makes no contribution in this fitted model. Its actual price is unknown, so this is not an evaluation result.

| Alpha | First coefficient | Second coefficient | Nonzero count |
|---|---:|---:|---:|
| 0 | 2 | 1 | 2 |
| 0.5 | 1.5 | 0.5 | 2 |
| 1 | 1 | 0 | 1 |
| 2 | 0 | 0 | 0 |

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Worked-example takeaway</strong><br>The second feature contributes to the exact training relationship, yet Lasso removes it at this alpha. A zero coefficient reflects a penalized fit, not proof that a feature has no underlying effect.</div>

The simple independent thresholding in this example relies on orthogonal columns. Real correlated features require joint optimization, and their selected sets need not change in such a simple sequence.

## 10. Geometry and the Bias–Variance Tradeoff

Lasso can also be described as minimizing squared error under a coefficient budget:

$$
\sum_{j=1}^{p}|w_j|\leq t,
$$

where $t\geq0$ is the allowed L1 total. For two coefficients, the boundary is a diamond with corners on the axes. A squared-error contour can first meet the allowed region at a corner, where one coefficient is zero. Ridge's corresponding region is round. This geometry helps explain why Lasso encourages sparsity.

A suitable budget can correspond to a penalized solution, but $t$ is not simply the reciprocal of alpha. The relationship depends on the data.

Regularization also changes prediction behavior across hypothetical repeated training samples. **Bias** is systematic error in the average fitted prediction; **variance** is how much predictions change across samples. Shrinking or removing coefficients introduces bias but can reduce variance and improve prediction on new data. Excessive regularization discards useful signal and underfits.

For fixed data, the total L1 norm of exact optimal solutions is nonincreasing as alpha grows, while training squared error is nondecreasing. Test error has no such guarantee. With correlated features, individual coefficients and the number of selected features need not change monotonically throughout the path.

## 11. Scaling and Feature Representation

Change floor area from square feet to thousands of square feet, and the coefficient needed for the same prediction becomes 1000 times larger. Its L1 penalty becomes 1000 times larger too. Without deliberate scaling, measurement units influence which features Lasso retains.

A common choice is standardization:

$$
z_{ij}=\frac{x_{ij}-\mu_j}{s_j},
$$

where $\mu_j$ and $s_j$ are the mean and standard deviation of feature $j$ estimated from training data. This makes a unit change represent one training standard deviation. It does not make all features equally informative.

Handle missing values with a suitable training-fitted strategy such as imputation, and inspect extreme observations: squared-error loss and ordinary standardization are both sensitive to outliers. Remove constant features or handle them explicitly since they contain no variation.

Use meaningful encodings for categorical variables. Arbitrary category numbers can imply an unjustified ordered linear effect. With one-hot encoding, Lasso may select individual category indicators rather than an entire original variable. Indicator scaling and the chosen reference category can affect the penalized fit, so document and validate those choices.

<div style="color:#22543d; background-color:#edf8f1; border-left:5px solid #2f855a; padding:10px 14px; margin:12px 0;"><strong>Practical tip</strong><br>Keep imputation, encoding, scaling, and Lasso inside the training pipeline. During cross-validation, fit every learned transformation only on that fold's training rows.</div>

Target units matter too. Multiplying all targets by a positive constant changes the relative balance between squared error and the L1 penalty at fixed alpha. Retune alpha when changing the target representation.

## 12. Correlated Features and Selection Stability

If two features represent nearly the same information, one can explain much of what the other explains. Lasso may keep one and set the other to zero. Small data changes can reverse that choice without greatly changing prediction performance.

For an exact example, let two feature columns be identical. Coefficient pairs $(2,0)$, $(1,1)$, and $(0,2)$ give the same predictions. They also have the same L1 norm of 2. The L1 penalty cannot distinguish these allocations. If their total effect is optimal, multiple coefficient solutions may be optimal too. A numerical solver can return one representative solution.

This contrasts with Ridge's squared penalty, which prefers the balanced allocation among these pairs. Elastic Net adds a squared penalty to the L1 penalty and can stabilize how correlated features share effects.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Selection is not discovery of truth</strong><br>A selected feature may be a proxy for another feature. A discarded feature may still carry useful signal. Check whether selections remain similar across training resamples before presenting them as a stable finding.</div>

Distinguish prediction stability from selection stability. A model may predict consistently while choosing different representatives of a correlated group. A compact feature list alone is not enough evidence of interpretability or causality.

## 13. Choosing Alpha and Checking Convergence

| Setting | What it controls | Practical consequence |
|---|---|---|
| `alpha` | Strength of L1 regularization | Balances fit, shrinkage, and sparsity; select using validation |
| `fit_intercept` | Whether to fit an unpenalized baseline | Usually retain unless the data setup justifies omitting it |
| `max_iter` | Maximum solver iterations | Increase if needed for convergence; it is not model complexity |
| `tol` | Solver stopping tolerance | Tighter accuracy may require more computation |
| `selection` | Coordinate update order | Can affect convergence speed and the returned solution when nonunique |

The companion allows 10,000 iterations. That is an iteration limit, not proof that every fit converged. Investigate convergence warnings, feature scaling, and solver diagnostics such as `n_iter_` and `dual_gap_`. The latter describes an optimization gap used to assess closeness to the optimum; it is not a test-error metric. These implementation details are described in the [Lasso API reference](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html).

For centered features and target under our convention, a useful upper threshold is

$$
\alpha_{\max}=\max_j\left|\frac{1}{n}\sum_i z_{ij}v_i\right|.
$$

At or above this value, all-zero feature coefficients satisfy the optimality conditions; the intercept predicts the training mean. This follows from the zero condition in soft thresholding with every other coefficient zero. Compute such a threshold from training data, never the test targets. It helps explain the scale of a search, but does not identify the best alpha.

A dense coordinate-descent pass can cost roughly $O(np)$ with maintained residuals; total cost also depends on the number of passes, sparsity, and solver strategy. Correlated features and weak regularization can make fitting slower. Sparse coefficients can simplify prediction, although standard pipelines may still require all original input columns.

## 14. A Reliable Training and Validation Workflow

1. Define the prediction target, available prediction-time features, and the main error metric. Exclude information created after the outcome.
2. Set aside a final test set. Use a time-respecting split for future prediction or a group-aware split for repeated subjects or related observations.
3. Build the preprocessing-and-Lasso pipeline. Within the development training data, compare alpha values with cross-validation: fit on each fold's training portion and score on its validation portion.
4. Try a broad, logarithmically spaced range, then refine where useful. The companion's $0.001,0.01,0.1,1,10$ are example candidates, not a universal range. Inspect variation across folds as well as the mean score.
5. Select alpha using validation performance. If sparsity has an operational value, make that tradeoff explicit rather than silently preferring the fewest features.
6. Refit the chosen full pipeline on all development training rows and evaluate once on the untouched test set.
7. Apply the fitted transformations and model to new rows with the same feature definitions and schema.

For the companion's pipeline names, a search over `model__alpha` tunes the Lasso step. Passing the whole pipeline to a cross-validation search refits scaling inside each fold. Scaling once before cross-validation would let validation rows influence training preprocessing.

Feature selection itself must also happen inside each fold. Selecting columns using all targets and then cross-validating a model on those columns leaks outcome information. If you refit an unpenalized model after Lasso selection, that is a different modeling procedure and needs its own complete validation.

## 15. Connecting the Lesson to the Code Companion

`04_Lasso_Regression.ipynb` loads `load_diabetes(as_frame=True)` and makes an 80/20 training/test split with `random_state=42`. It fits three pipelines, each with `StandardScaler`: ordinary Linear Regression, `Lasso(alpha=1.0, max_iter=10000)`, and `ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)`.

It compares test MAE, RMSE, and $R^2$, then inspects Lasso's coefficients. The `selected_by_lasso` column uses `coef_ != 0`, which reports the returned model's nonzero coefficients. A tiny nonzero value still meets that definition; practical importance and numerical accuracy require interpretation beyond this Boolean flag.

The final loop fits Lasso at five alpha values and reports test scores and selected-feature counts. It illustrates how models differ; these notes do not claim any particular scores, selected features, or winner because the companion has not been executed for this revision.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Keep the test set independent</strong><br>The companion evaluates its alpha sweep on the same test set. Choosing the best alpha from that table would use the test set for tuning. For model selection, move the comparison into training-only cross-validation and reserve the test set for the final assessment.</div>

The fixed alpha values are examples, not established optima. Comparing the same numeric alpha across Lasso and Elastic Net also does not isolate an identical regularization effect.

## 16. Evaluate Predictions and Interpret Coefficients

On $m$ held-out rows, define residuals $e_i=y_i-\hat y_i$. Common metrics are

$$
\mathrm{MAE}=\frac{1}{m}\sum_i|e_i|,\qquad
\mathrm{MSE}=\frac{1}{m}\sum_i e_i^2,\qquad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}}.
$$

MAE is average absolute error in target units. RMSE also uses target units but gives large errors more influence. Lower is better on the same evaluation data. Do not add the L1 penalty to these held-out prediction metrics.

For nonconstant evaluation targets,

$$
R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y_{\mathrm{eval}})^2},
$$

where $\bar y_{\mathrm{eval}}$ is their mean. One is perfect, zero matches the evaluation-mean reference in this formula, and negative values are worse. Constant targets make this mathematical ratio undefined. Also compare with a deployable baseline trained to predict the training target mean.

Inspect residual patterns, large errors, and performance across relevant groups. A few selected features and a low training error do not establish good generalization.

For standardized feature $j$, coefficient $w_j$ is the change in prediction for a one-training-standard-deviation increase, holding the other represented features fixed. Recover its raw-unit coefficient as $\beta_j=w_j/s_j$. With standardized intercept $b_z$, the equivalent raw intercept is

$$
b_{\mathrm{raw}}=b_z-\sum_j\frac{w_j\mu_j}{s_j}.
$$

These formulas assume the target was not transformed. Lasso deliberately shrinks coefficients, so their magnitudes are biased estimates of unpenalized effects. Holding correlated features fixed may also describe unrealistic cases. Coefficients express predictive associations, not causal effects, and ordinary regression significance tests do not automatically remain valid after data-driven selection.

## 17. Ridge, Lasso, Elastic Net, and Other Choices

| Method | Penalty idea | Typical reason to compare it |
|---|---|---|
| Ordinary least squares | None | A simple baseline when fitting is stable |
| Ridge | Squared coefficient magnitudes | Many features may contribute; correlated effects need stabilization |
| Lasso | Absolute coefficient magnitudes | A sparse linear representation is plausible or useful |
| Elastic Net | L1 plus squared L2 | Sparsity with more stable handling of correlated predictors |
| Polynomial features plus Lasso | L1 on an expanded feature set | Select among proposed powers or interactions |
| Trees or forests | Split-based models rather than these penalties | Important nonlinearities or interactions are not well represented linearly |

To connect with the original notes, a generic Elastic Net objective can be written

$$
\tfrac12\mathrm{MSE}+\lambda_1\sum_j|w_j|+\lambda_2\sum_jw_j^2,
$$

where $\lambda_1$ and $\lambda_2$ are nonnegative weights for the two penalties. Increasing the L1 component encourages sparsity; adding a positive squared component helps distinguish otherwise interchangeable coefficient allocations. Library parameterizations combine these weights differently, so consult their full objective before transferring numeric settings.

Lasso is a reasonable candidate when a small subset of represented features may explain most of the signal. Ridge may be preferable when many modest contributions matter. These are hypotheses to evaluate, not guarantees. With transformed features, sparsity applies to individual columns: Lasso can retain a squared term while discarding its linear term unless additional structure is imposed.

## 18. Assumptions, Limitations, and Recap

Lasso can be fitted without normally distributed errors. Useful predictions still require informative features, a suitable representation of the target relationship, and training data relevant to future cases. Independent rows support ordinary random-split evaluation; dependent observations require an appropriate validation design.

| Failure mode | Why it matters | Response |
|---|---|---|
| Excessive regularization | Useful effects are removed or biased too strongly | Revisit alpha using validation |
| Strong feature correlation | Selected columns may be unstable or coefficients nonunique | Inspect resampling stability and compare Elastic Net or Ridge |
| Nonlinear structure omitted | A sparse linear equation still misses the pattern | Improve representations or compare nonlinear models |
| Outliers or poor measurements | Squared error can give unusual rows large influence | Investigate data quality and robust alternatives |
| Leakage or distribution shift | Reported performance may not reflect future prediction | Audit features, splits, and later performance |
| Dense underlying signal | Removing many small effects can hurt predictions | Compare a model that retains them, such as Ridge |

Lasso can extrapolate numerically beyond observed targets, but the linear equation does not make that extrapolation reliable. It also does not guarantee recovery of the true relevant variables; that requires much stronger conditions than simply choosing a positive alpha.

The central idea is a linear prediction rule fitted with an absolute-value penalty. The corner at zero creates soft thresholding, which explains both coefficient shrinkage and exact zeros. The main choices are representation, scaling, penalty strength, and validation design.

Remember the worked example: the removed feature helped describe the data, but the selected alpha made its contribution too costly. A sparse model is an outcome to assess, not proof that the retained features are the only ones that matter.